In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
%reload_ext autoreload

In [3]:
import random
import numpy as np
import torch
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

In [4]:
import data
# data.download_pipeline() # 다운로드 필요시
# data._unzip_downloaded_data() # 압축해재만
df = data.preprocess_pipeline()

preprocess_pipeline: _to_pandas_table done..
preprocess_pipeline: _preprocess_pandas_table done..
preprocess_pipeline: _add_date_features done..
preprocess_pipeline: _add_lag_and_rolling_features done..
preprocess_pipeline: all process done..


In [5]:
df.info()
df.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 101848 entries, 0 to 101847
Data columns (total 21 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   접수일자                 101848 non-null  datetime64[us]
 1   접수지역                 101848 non-null  category      
 2   배달지역                 101848 non-null  category      
 3   총중량_kg               101848 non-null  float64       
 4   연도                   101848 non-null  int32         
 5   월                    101848 non-null  int32         
 6   일                    101848 non-null  int32         
 7   요일                   101848 non-null  int32         
 8   공휴일여부                101848 non-null  int64         
 9   다음_공휴일까지_남은일수        101848 non-null  int64         
 10  지난_공휴일부터_지난일수        101848 non-null  int64         
 11  공휴일_직전_3일내           101848 non-null  int64         
 12  공휴일_직후_2일내           101848 non-null  int64         
 13  lag_1                1018

접수일자                   0
접수지역                   0
배달지역                   0
총중량_kg                 0
연도                     0
월                      0
일                      0
요일                     0
공휴일여부                  0
다음_공휴일까지_남은일수          0
지난_공휴일부터_지난일수          0
공휴일_직전_3일내             0
공휴일_직후_2일내             0
lag_1                  0
lag_2                  0
lag_7                  0
lag_14                 0
rolling_7_mean         0
rolling_7_std          0
rolling_14_mean        0
ratio_lag1_to_roll7    0
dtype: int64

In [6]:
df_train, df_test = data.split_timeseries_data(df)

📊 시계열 데이터셋 분할 완료
 전체 데이터 기간: 2025-01-20 ~ 2026-06-30
 Train 데이터   : 2025-01-20 ~ 2026-05-29 (95,776건)
 Test 데이터    : 2026-06-01 ~ 2026-06-30 (6,072건)


In [7]:
# ⚠️ 수정: LGBM의 early stopping이 df_test를 직접 보면 테스트 성능이 낙관적으로 부풀려짐(누수).
# 그래서 훈련기간(df_train) 내부에서 검증셋을 별도로 떼어낸다. df_test는 이 시점 이후로
# '최종 평가' 딱 한 번에만 사용하고, 어떤 학습/튜닝 결정에도 관여시키지 않는다.
df_train2, df_valid = data.split_timeseries_data(df_train, test_days=30)

📊 시계열 데이터셋 분할 완료
 전체 데이터 기간: 2025-01-20 ~ 2026-05-29
 Train 데이터   : 2025-01-20 ~ 2026-04-29 (90,241건)
 Test 데이터    : 2026-04-30 ~ 2026-05-29 (5,535건)


In [8]:
import lgbm, models

# 학습: df_train2로 학습하고, df_valid로 early stopping 기준을 잡는다 (df_test는 안 씀)
model_lgbm = lgbm.train_lgbm(df_train2, df_valid)

# 검증셋 예측 (앙상블 가중치 탐색용) + 테스트셋 예측 (최종 평가용)
preds_lgbm_valid = lgbm.predict_lgbm(model_lgbm, df_valid)
preds_lgbm_test = lgbm.predict_lgbm(model_lgbm, df_test)

print("[LGBM] 검증셋 성능:")
models.evaluate_outputs(df_valid["총중량_kg"], preds_lgbm_valid)
print("[LGBM] 테스트셋 성능 (최종, 한 번만 확인):")
models.evaluate_outputs(df_test["총중량_kg"], preds_lgbm_test)

LightGBM 모델 학습 시작...


/home/oudedong/postoffice/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004486 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2131
[LightGBM] [Info] Number of data points in the train set: 90241, number of used features: 14
[LightGBM] [Info] Start training from score 3596.625657
[LGBM] 검증셋 성능:

📈 모델 평가 결과
 MAE  (평균 절대 오차) : 772.92 kg
 RMSE (평균 제곱근 오차): 1,955.80 kg
 WAPE (가중 오차율)    : 25.96 %
[LGBM] 테스트셋 성능 (최종, 한 번만 확인):

📈 모델 평가 결과
 MAE  (평균 절대 오차) : 639.97 kg
 RMSE (평균 제곱근 오차): 1,434.60 kg
 WAPE (가중 오차율)    : 18.07 %


In [9]:
import lstm

# 1. 학습 (df_train2만 사용 — df_valid/df_test는 학습에 관여하지 않음)
model_lstm, label_encoders, scaler = lstm.train_lstm(df_train2, epochs=30)

# 2. 검증셋 예측 (앙상블 가중치 탐색용)
preds_lstm_valid, valid_target_indices = lstm.predict_lstm(
    model_lstm, df_valid, label_encoders, scaler
)
y_valid_matched = df_valid.loc[valid_target_indices, "총중량_kg"].values

# 3. 테스트셋 예측 (최종 평가용)
preds_lstm_test, test_target_indices = lstm.predict_lstm(
    model_lstm, df_test, label_encoders, scaler
)
y_test_matched = df_test.loc[test_target_indices, "총중량_kg"].values

print("[LSTM] 검증셋 성능:")
models.evaluate_outputs(y_valid_matched, preds_lstm_valid)
print("[LSTM] 테스트셋 성능 (최종, 한 번만 확인):")
models.evaluate_outputs(y_test_matched, preds_lstm_test)

🖥️ 사용 장치: cuda

🚀 Categorical Embedding + LSTM 모델 학습 시작...
Epoch [01/30] | Loss (Huber): 0.04285
Epoch [05/30] | Loss (Huber): 0.02161
Epoch [10/30] | Loss (Huber): 0.01653
Epoch [15/30] | Loss (Huber): 0.01359
Epoch [20/30] | Loss (Huber): 0.01114
Epoch [25/30] | Loss (Huber): 0.01000
Epoch [30/30] | Loss (Huber): 0.00921
[LSTM] 검증셋 성능:

📈 모델 평가 결과
 MAE  (평균 절대 오차) : 550.91 kg
 RMSE (평균 제곱근 오차): 1,506.67 kg
 WAPE (가중 오차율)    : 18.48 %
[LSTM] 테스트셋 성능 (최종, 한 번만 확인):

📈 모델 평가 결과
 MAE  (평균 절대 오차) : 496.72 kg
 RMSE (평균 제곱근 오차): 1,281.57 kg
 WAPE (가중 오차율)    : 12.66 %


In [10]:
import pandas as pd

# LGBM 예측은 df_valid/df_test 전체 인덱스 기준이고, LSTM 예측은 SEQ_LEN(14)만큼
# 앞부분이 빠진 target_indices 기준이라 인덱스를 맞춰서 비교해야 함.

# 1. 검증셋에서 최적 앙상블 가중치 탐색 (테스트셋은 여기 전혀 관여 안 함)
preds_lgbm_valid_matched = pd.Series(preds_lgbm_valid, index=df_valid.index).loc[valid_target_indices].values

w_lstm, w_lgb = models.find_best_ensemble_weight(
    y_valid_matched, preds_lstm_valid, preds_lgbm_valid_matched
)

# w_lstm, w_lgb = 0.585, 0.415 # 원래 쓰던거

# 2. 탐색한 가중치를 테스트셋에 '그대로' 적용해서 최종 평가 (여기서 가중치를 다시 조정하면 또 누수)
preds_lgbm_test_matched = pd.Series(preds_lgbm_test, index=df_test.index).loc[test_target_indices].values
final_pred = w_lstm * preds_lstm_test + w_lgb * preds_lgbm_test_matched

print("[앙상블] 최종 테스트셋 성능:")
models.evaluate_outputs(y_test_matched, final_pred)


🔍 검증셋 기준 최적 앙상블 가중치: w_lstm=1.00, w_lgb=0.00 (valid WAPE=18.48%)
[앙상블] 최종 테스트셋 성능:

📈 모델 평가 결과
 MAE  (평균 절대 오차) : 496.72 kg
 RMSE (평균 제곱근 오차): 1,281.57 kg
 WAPE (가중 오차율)    : 12.66 %


In [21]:
import dispatch, integration
import pandas as pd

# 1. 앞서 모델로 예측한 DataFrame 준비 (컬럼: 접수일자, 접수지역, 배달지역, 예측중량_kg)
#    (예시: df_test에 final_pred 예측값을 '예측중량_kg' 컬럼으로 추가한 상태)
df_pred = df_test.copy()
# 1. 인덱스를 지정한 Series를 할당 (매칭 안 되는 행은 자동으로 NaN으로 채워짐)
df_pred["예측중량_kg"] = pd.Series(preds_lstm_test, index=test_target_indices)
# 2. NaN인 행(과거 14일치 히스토리용 앞부분) 제거
df_pred = df_pred.dropna(subset=["예측중량_kg"])
# 2. 날짜별 x 목적지별 전체 배차 최적화 돌리기
summary_df = integration.run_daily_dispatch_optimization(
    df_pred=df_pred,
    date_col="접수일자",
    origin_col="접수지역",
    dest_col="배달지역",
    weight_col="예측중량_kg",
)

# 3. 최적화 요약 테이블 확인 (날짜/목적지별 직송비용, 허브비용, 절감액 등)
print(summary_df.head())

# 4. 전체 기간 최종 절감률(%) 리포트 출력
summary_stats = integration.summarize_savings(summary_df)
integration.print_savings_report(summary_stats)

          날짜      목적지  출발지_수         총물량_kg    직송_비용  허브경유_최적비용 선택된_허브  \
0 2026-06-19    울산광역시      1    2036.089174    94491      94491    NaN   
1 2026-06-19    인천광역시      1    3423.587017   158088     158088    NaN   
2 2026-06-22  강원특별자치도     17   45307.966130  2393690    2189379  대구광역시   
3 2026-06-22      경기도     17  358661.822368  8161570    8077698   충청북도   
4 2026-06-22     경상남도     17   70489.710238  2714915    2601256  대전광역시   

    최종_결정    최종_비용     절감액  
0  direct    94491       0  
1  direct   158088       0  
2     hub  2189379  204311  
3     hub  8077698   83872  
4     hub  2601256  113659  
📊 배차 최적화 결과 요약 (예측기간 전체)
 직송 기준(baseline) 총비용   : 320,776,389 원
 우리 모델(예측+허브판단) 총비용: 301,763,415 원
 총 절감액                  : 19,012,974 원
 절감률                    : 5.93%
 허브경유 선택 건수         : 119 / 122 (97.54%)


In [14]:
import dispatch, integration
import pandas as pd

# 1. 앞서 모델로 예측한 DataFrame 준비 (컬럼: 접수일자, 접수지역, 배달지역, 예측중량_kg)
#    (예시: df_test에 final_pred 예측값을 '예측중량_kg' 컬럼으로 추가한 상태)
df_pred = df_test.copy()
# 1. 인덱스를 지정한 Series를 할당 (매칭 안 되는 행은 자동으로 NaN으로 채워짐)
df_pred["예측중량_kg"] = pd.Series(preds_lstm_test, index=test_target_indices)
# 2. NaN인 행(과거 14일치 히스토리용 앞부분) 제거
df_pred = df_pred.dropna(subset=["예측중량_kg"])
# 2. 날짜별 x 목적지별 전체 배차 최적화 돌리기
summary_df2 = integration.run_daily_dispatch_optimization(
    df_pred=df_pred,
    date_col="접수일자",
    origin_col="접수지역",
    dest_col="배달지역",
    weight_col="예측중량_kg",
)

# 3. 최적화 요약 테이블 확인 (날짜/목적지별 직송비용, 허브비용, 절감액 등)
print(summary_df2.head())

# 4. 전체 기간 최종 절감률(%) 리포트 출력
summary_stats2 = integration.summarize_savings(summary_df2)
integration.print_savings_report(summary_stats2)

          날짜      목적지  출발지_수         총물량_kg    직송_비용  허브경유_최적비용 선택된_허브  \
0 2026-06-19    울산광역시      1    2036.089174    94491      94491    NaN   
1 2026-06-19    인천광역시      1    3423.587017   158088     158088    NaN   
2 2026-06-22  강원특별자치도     17   45307.966130  2393690    2189379  대구광역시   
3 2026-06-22      경기도     17  358661.822368  8161570    8077698   충청북도   
4 2026-06-22     경상남도     17   70489.710238  2714915    2601256  대전광역시   

    최종_결정    최종_비용     절감액  
0  direct    94491       0  
1  direct   158088       0  
2     hub  2189379  204311  
3     hub  8077698   83872  
4     hub  2601256  113659  
📊 배차 최적화 결과 요약 (예측기간 전체)
 직송 기준(baseline) 총비용   : 320,776,389 원
 우리 모델(예측+허브판단) 총비용: 301,763,415 원
 총 절감액                  : 19,012,974 원
 절감률                    : 5.93%
 허브경유 선택 건수         : 119 / 122 (97.54%)
